In [2]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 2. LOAD DATA

In [4]:
data = pd.read_csv("/content/trum_tweet_sentiment_analysis.csv")

#3. CLEANING FUNCTIONS

In [5]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [6]:
def convert_to_string(text):
    return str(text)

In [7]:
def to_lowercase(text):
    return text.lower()

In [8]:
def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

In [9]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

In [10]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [11]:
def remove_extra_spaces(text):
    return text.strip()

In [12]:

def tokenize(text):
    return text.split()

In [13]:
def remove_stopwords(words):
    return [word for word in words if word not in stop_words]

In [14]:
def apply_stemming(words):
    return [stemmer.stem(word) for word in words]

In [15]:
def join_words(words):
    return " ".join(words)

# 4. MAIN CLEANING PIPELINE

In [16]:
def clean_text(text):
    text = convert_to_string(text)
    text = to_lowercase(text)
    text = remove_urls(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)

    words = tokenize(text)
    words = remove_stopwords(words)
    words = apply_stemming(words)

    return join_words(words)

#5. APPLY CLEANING

In [17]:
data['clean_text'] = data['text'].apply(clean_text)

In [23]:
data = data[data['clean_text'].str.strip() != ""].dropna(subset=['Sentiment'])

#6. TRAIN-TEST SPLIT

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'],
    data['Sentiment'],
    test_size=0.2,
    random_state=42
)

#7. FEATURE EXTRACTION (TF-IDF)

In [25]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# 8. MODEL TRAINING

In [26]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

#9. EVALUATION

In [27]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9283250968819995

Classification Report:

              precision    recall  f1-score   support

         0.0       0.93      0.96      0.95     18441
         1.0       0.92      0.86      0.89      9686

    accuracy                           0.93     28127
   macro avg       0.93      0.91      0.92     28127
weighted avg       0.93      0.93      0.93     28127



#10. PREDICTION FUNCTION

In [28]:
def predict_text(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)
    return prediction[0]

#11. TEST EXAMPLES

In [29]:
samples = [
    "This product is amazing and works perfectly!",
    "Worst service ever, I hate it"
]

for s in samples:
    print("\nText:", s)
    print("Prediction:", predict_text(s))


Text: This product is amazing and works perfectly!
Prediction: 1.0

Text: Worst service ever, I hate it
Prediction: 0.0
